<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/05_applications/resume_skill_extraction_and_weighted_scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers scikit-learn pandas

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import re

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
priority_skills = [
    "machine learning",
    "deep learning",
    "nlp",
    "python",
    "data science"
]

In [5]:
job_description = """
We are hiring a machine learning engineer with strong experience in
Python, NLP, deep learning, and building intelligent systems.
"""

In [6]:
resumes = [
    """
    Data scientist with experience in machine learning, NLP, and Python.
    Built semantic search systems and worked on deep learning models.
    """,

    """
    Frontend developer skilled in HTML, CSS, JavaScript, and React.
    Focused on UI development and web performance.
    """,

    """
    Machine learning engineer with expertise in Python, deep learning,
    NLP pipelines, and AI-based recommendation systems.
    """,

    """
    Software engineer experienced in Java, SQL, and backend systems.
    Worked on database optimization and APIs.
    """
]

In [7]:
def extract_skills(text, skill_list):
    text = text.lower()
    found_skills = []
    for skill in skill_list:
        if re.search(rf"\b{skill}\b", text):
            found_skills.append(skill)
    return found_skills

In [8]:
job_skills = extract_skills(job_description, priority_skills)

resume_skills = [
    extract_skills(resume, priority_skills)
    for resume in resumes
]

job_skills, resume_skills

(['machine learning', 'deep learning', 'nlp', 'python'],
 [['machine learning', 'deep learning', 'nlp', 'python'],
  [],
  ['machine learning', 'deep learning', 'nlp', 'python'],
  []])

In [9]:
job_embedding = model.encode([job_description])
resume_embeddings = model.encode(resumes)

In [10]:
semantic_scores = cosine_similarity(job_embedding, resume_embeddings)[0]
semantic_scores

array([0.6973975 , 0.34983933, 0.7581085 , 0.51140726], dtype=float32)

In [11]:
skill_scores = []

for skills in resume_skills:
    if len(job_skills) == 0:
        skill_scores.append(0)
    else:
        matched = len(set(skills) & set(job_skills))
        skill_scores.append(matched / len(job_skills))

skill_scores

[1.0, 0.0, 1.0, 0.0]

In [12]:
final_scores = []

for sem, skill in zip(semantic_scores, skill_scores):
    score = (0.7 * sem) + (0.3 * skill)
    final_scores.append(score)

final_scores

[np.float32(0.7881782),
 np.float32(0.24488753),
 np.float32(0.83067596),
 np.float32(0.35798508)]

In [13]:
results = pd.DataFrame({
    "Resume Index": range(len(resumes)),
    "Semantic Score": semantic_scores,
    "Skill Match Score": skill_scores,
    "Final Score": final_scores
})

results = results.sort_values(by="Final Score", ascending=False)
results

,Resume Index,Semantic Score,Skill Match Score,Final Score
2,2,0.758108,1.0,0.830676
0,0,0.697397,1.0,0.788178
3,3,0.511407,0.0,0.357985
1,1,0.349839,0.0,0.244888


In [16]:
for index, row in results.iterrows():
    i = int(row["Resume Index"])
    print("Resume Index:", i)
    print("Semantic Score:", round(row["Semantic Score"], 3))
    print("Skill Match Score:", round(row["Skill Match Score"], 3))
    print("Final Score:", round(row["Final Score"], 3))
    print("Matched Skills:", resume_skills[i])
    print("Resume Text:", resumes[i])
    print("-" * 70)

Resume Index: 2
Semantic Score: 0.758
Skill Match Score: 1.0
Final Score: 0.831
Matched Skills: ['machine learning', 'deep learning', 'nlp', 'python']
Resume Text: 
    Machine learning engineer with expertise in Python, deep learning,
    NLP pipelines, and AI-based recommendation systems.
    
----------------------------------------------------------------------
Resume Index: 0
Semantic Score: 0.697
Skill Match Score: 1.0
Final Score: 0.788
Matched Skills: ['machine learning', 'deep learning', 'nlp', 'python']
Resume Text: 
    Data scientist with experience in machine learning, NLP, and Python.
    Built semantic search systems and worked on deep learning models.
    
----------------------------------------------------------------------
Resume Index: 3
Semantic Score: 0.511
Skill Match Score: 0.0
Final Score: 0.358
Matched Skills: []
Resume Text: 
    Software engineer experienced in Java, SQL, and backend systems.
    Worked on database optimization and APIs.
    
---------------

- This notebook enhances resume-job matching by combining semantic similarity with skill-based scoring.
- Sentence embeddings are used to compute semantic relevance, while keyword-based skill extraction adds explainability and control.
- A weighted scoring strategy is applied to rank resumes more realistically.